In [4]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PATH = "/content/drive/MyDrive/food-pipeline"
MODEL_SAVE_PATH = f"{DRIVE_PATH}/models/classifier"
DATA_PATH = f"{DRIVE_PATH}/data/food101"

import os
os.makedirs(MODEL_SAVE_PATH, exist_ok=True)
os.makedirs(DATA_PATH, exist_ok=True)
print("Drive mounted and paths ready")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted and paths ready


In [5]:
!pip install -q timm safetensors huggingface_hub wandb

In [6]:
import torch
print(f"GPU available: {torch.cuda.is_available()}")
print(f"GPU name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

GPU available: True
GPU name: Tesla T4


In [7]:
from huggingface_hub import hf_hub_download
import json

hf_hub_download(
    repo_id="Lumia101/Food101-EfficientNet-B0",
    filename="config.json",
    local_dir=MODEL_SAVE_PATH
)
hf_hub_download(
    repo_id="Lumia101/Food101-EfficientNet-B0",
    filename="model.safetensors",
    local_dir=MODEL_SAVE_PATH
)
print("Base model downloaded")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/19.1M [00:00<?, ?B/s]

Base model downloaded


In [8]:
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

train_dataset = torchvision.datasets.Food101(
    root=DATA_PATH, split='train',
    transform=train_transform, download=True
)
val_dataset = torchvision.datasets.Food101(
    root=DATA_PATH, split='test',
    transform=val_transform, download=False
)

train_loader = DataLoader(train_dataset, batch_size=64,
                          shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=64,
                        shuffle=False, num_workers=2, pin_memory=True)

print(f"Train: {len(train_dataset)} images")
print(f"Val:   {len(val_dataset)} images")
print(f"Classes: {len(train_dataset.classes)}")

100%|██████████| 5.00G/5.00G [03:00<00:00, 27.7MB/s] 


Train: 75750 images
Val:   25250 images
Classes: 101


In [9]:
import torch
import torch.nn as nn
import torchvision.models as models
from safetensors.torch import load_file

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.efficientnet_b0(weights=None)
model.classifier = nn.Sequential(
    nn.Dropout(p=0.2, inplace=True),
    nn.Linear(1280, 512),
    nn.SiLU(),
    nn.Dropout(0.2),
    nn.Linear(512, 101)
)

state_dict = load_file(f"{MODEL_SAVE_PATH}/model.safetensors")
model.load_state_dict(state_dict)
model = model.to(device)
print(f"Model loaded on {device}")

Model loaded on cuda


In [10]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import time

optimizer = AdamW([
    {"params": model.features.parameters(), "lr": 5e-5},
    {"params": model.classifier.parameters(), "lr": 2e-4}
])
scheduler = CosineAnnealingLR(optimizer, T_max=10)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += outputs.argmax(1).eq(labels).sum().item()
        total += labels.size(0)
    return total_loss / len(loader), correct / total

def val_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            correct += outputs.argmax(1).eq(labels).sum().item()
            total += labels.size(0)
    return total_loss / len(loader), correct / total

EPOCHS = 10
best_acc = 0.7957  # start tracking above Lumia101 baseline

for epoch in range(EPOCHS):
    t0 = time.time()
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc = val_epoch(model, val_loader, criterion, device)
    scheduler.step()

    elapsed = time.time() - t0
    print(f"Epoch {epoch+1:02d}/{EPOCHS} | "
          f"train_loss={train_loss:.3f} acc={train_acc:.3f} | "
          f"val_loss={val_loss:.3f} acc={val_acc:.3f} | "
          f"{elapsed:.0f}s")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(),
                   f"{MODEL_SAVE_PATH}/efficientnet_b0_food101_best.pt")
        print(f"  Saved new best model (acc={best_acc:.4f})")

KeyboardInterrupt: 

In [ ]:
import json

class_to_idx = train_dataset.class_to_idx
idx_to_class = {v: k for k, v in class_to_idx.items()}

with open(f"{MODEL_SAVE_PATH}/idx_to_class.json", "w") as f:
    json.dump(idx_to_class, f, indent=2)

print(f"Saved {len(idx_to_class)} class labels")
print("Sample:", list(idx_to_class.items())[:5])